# PyTorch 网络层详解 - 容器

本 Notebook 是系列教程的第4部分，详细介绍 PyTorch 中用于组织和管理网络组件的各种容器。

**包含内容**：
- 容器核心概念
- `nn.Sequential` - 顺序容器
- `nn.ModuleList` - 列表容器
- `nn.ModuleDict` - 字典容器
- `nn.ParameterList` - 参数列表容器
- `nn.ParameterDict` - 参数字典容器
- Python 原生列表 vs 容器的对比

In [1]:
# ============================================================
# 1. 导入必要的库
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子，保证结果可重复
torch.manual_seed(42)

# 打印 PyTorch 版本信息
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.7.1+cu118


## 1. 容器核心概念

**容器（Container）** 在PyTorch中是指**用于组织和管理其他模块/参数的数据结构**。

### 1.1 PyTorch 提供的容器类

PyTorch 在 `torch.nn` 中提供了 5 种专用容器类，它们都继承自 `nn.Module`：

| 容器类             | 用途                       | 类比（对比式）                                   |
| :----------------- | :------------------------- | :----------------------------------------------- |
| `nn.Sequential`    | 按顺序执行网络层           | **自动化流水线**（不用你管，放进去自动加工）     |
| `nn.ModuleList`    | 像列表一样存储多个模块     | **工具箱**（工具按顺序放，但怎么用由你决定）     |
| `nn.ModuleDict`    | 像字典一样通过键名管理模块 | **带标签的抽屉柜**（按标签找工具，不用翻）       |
| `nn.ParameterList` | 存储多个可训练参数         | **零件盒**（装螺丝钉、螺母，按顺序取用）         |
| `nn.ParameterDict` | 通过名称管理可训练参数     | **带标签的零件盒**（每个格子有标签，不用记位置） |
### 1.2 所有容器的共同特性——可迭代性

**所有 5 种容器都支持迭代**，这意味着你可以用 `for ... in` 遍历它们。但迭代返回的内容因容器类型而异：

| 容器 | 迭代方式 | 迭代返回 | 访问方式 |
|------|---------|---------|---------|
| `nn.Sequential` | 直接迭代 | 子模块（按顺序） | 索引 `[i]` |
| `nn.ModuleList` | 直接迭代 | 子模块（按顺序） | 索引 `[i]` |
| `nn.ModuleDict` | 默认迭代键 | 键名 | 键 `['key']` |
| `nn.ParameterList` | 直接迭代 | 参数（按顺序） | 索引 `[i]` |
| `nn.ParameterDict` | 默认迭代键 | 键名 | 键 `['key']` |

**迭代方式分类**：
- **列表风格**（`Sequential`、`ModuleList`、`ParameterList`）：`for x in container` 直接迭代元素
- **字典风格**（`ModuleDict`、`ParameterDict`）：`for key in container` 默认迭代键名，配合 `.items()` 迭代键值对

### 1.3 使用容器的必要性

1. **自动注册**：容器内的子模块和参数会自动注册到父模块
2. **设备管理**：调用 `.to(device)` 时，容器内所有内容自动迁移
3. **状态管理**：`.state_dict()` 和 `.load_state_dict()` 自动处理所有参数
4. **梯度管理**：所有参数自动参与梯度计算
5. **层次化组织**：构建清晰的网络结构
6. **代码可读性**：一眼看出网络层次结构
7. **可迭代性**：所有容器都支持迭代，便于循环处理和调试

### 1.4 容器类型完整对比

| 容器类型 | 功能 | 访问方式 | 迭代返回 | 适用场景 |
|---------|------|---------|---------|---------|
| `nn.Sequential` | 按顺序执行网络层 | 索引（整数） | 子模块 | 简单前馈网络 |
| `nn.ModuleList` | 像 Python 列表一样存储模块 | 索引（整数） | 子模块 | 动态循环网络 |
| `nn.ModuleDict` | 像 Python 字典一样通过键名管理模块 | 键（字符串） | 键名 | 多分支/多任务网络 |
| `nn.ParameterList` | 专门存储可训练参数的列表 | 索引（整数） | 参数 | 自定义参数管理 |
| `nn.ParameterDict` | 通过名称管理可训练参数的字典 | 键（字符串） | 键名 | 带名称的参数集 |

## 2. 容器 vs Python 原生数据结构

**一句话核心区别**：PyTorch 容器继承自 `nn.Module`，具备自动注册、设备迁移、状态管理等功能；Python 原生 list/dict 不具备这些功能。

| 数据类型 | 自动注册 | 有 `forward` | 动态修改 | 被 `parameters()` 识别 |
|---------|---------|-------------|---------|----------------------|
| Python `list` | ❌ | ❌ | ✅ | ❌ |
| Python `dict` | ❌ | ❌ | ✅ | ❌ |
| `nn.Sequential` | ✅ | ✅ | ✅ | ✅ |
| `nn.ModuleList` | ✅ | ❌ | ✅ | ✅ |
| `nn.ModuleDict` | ✅ | ❌ | ✅ | ✅ |

```python
# ❌ 错误：用原生 list/dict 存模块，参数不会注册
self.layers = [nn.Linear(10, 20), nn.Linear(20, 5)]

# ✅ 正确：用 PyTorch 容器
self.layers = nn.ModuleList([nn.Linear(10, 20), nn.Linear(20, 5)])
```

## 3. 五种容器的递进式讲解

### 3.1 `nn.Sequential`——最基础的容器

**最基础的容器是 `nn.Sequential`**，它解决了深度学习中最常见的需求：**按顺序执行一系列网络层**。

**优点**：
- ✅ 自带 `forward` 方法，数据自动按顺序流过各层
- ✅ 代码最简洁，一行 `model(x)` 完成前向传播
- ✅ 支持索引访问、迭代、获取长度
- ✅ 支持动态修改（`append`、`insert`、`pop`）

`nn.Sequential` 代码最简洁，但数据只能固定顺序单向流动，无法实现条件分支、跳跃连接等复杂逻辑。

**Sequential 的典型用法**：

In [2]:
# ============================================================
# 3.1.1 nn.Sequential - 基础使用
# ============================================================

# 创建 Sequential 模型，数据会按顺序依次流过每个层
seq_model = nn.Sequential(
    nn.Linear(10, 20),    # 第1层：全连接，输入10维，输出20维
    nn.ReLU(),            # 第2层：ReLU 激活函数
    nn.Linear(20, 5),     # 第3层：全连接，输入20维，输出5维
    nn.Softmax(dim=1)     # 第4层：Softmax 输出概率分布
)

# 创建随机输入数据，4个样本，每个样本10维
x = torch.randn(4, 10)

# 前向传播：数据依次经过所有层 —— 一行代码搞定！
output = seq_model(x)

# 打印输入输出形状变化
print(f"Sequential: {x.shape} -> {output.shape}")

Sequential: torch.Size([4, 10]) -> torch.Size([4, 5])


Sequential 支持动态增删改操作

In [3]:
# ============================================================
# 3.1.2 Sequential 支持动态修改（验证）
# ============================================================

# 创建一个简单的 Sequential
seq = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU()
)

print(f"初始长度: {len(seq)}")
print(f"初始结构: {seq}")


初始长度: 2
初始结构: Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
)


In [4]:
# 尝试 append
try:
    seq.append(nn.Linear(20, 5))
    print(f"✅ append 成功，新长度: {len(seq)}")
    print(f"当前结构: {seq}")
except Exception as e:
    print(f"❌ append 失败: {e}")

✅ append 成功，新长度: 3
当前结构: Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=5, bias=True)
)


In [5]:
# 尝试 insert
try:
    seq.insert(0, nn.Dropout(0.5))
    print(f"✅ insert 成功，新长度: {len(seq)}")
    print(f"新结构: {seq}")
except Exception as e:
    print(f"❌ insert 失败: {e}")


✅ insert 成功，新长度: 4
新结构: Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=10, out_features=20, bias=True)
  (2): ReLU()
  (3): Linear(in_features=20, out_features=5, bias=True)
)


In [6]:
# 尝试 pop
try:
    removed = seq.pop(0)
    print(f"✅ pop 成功，新长度: {len(seq)}，移除: {removed}")
    print(f"新结构: {seq}")
except Exception as e:
    print(f"❌ pop 失败: {e}")

✅ pop 成功，新长度: 3，移除: Dropout(p=0.5, inplace=False)
新结构: Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=5, bias=True)
)


In [7]:
# 尝试 extend：批量添加多个层
try:
    seq.extend([nn.ReLU(), nn.Linear(30, 5)])
    print(f"✅ pop 成功，新长度: {len(seq)}")
    print(f"新结构: {seq}")
except Exception as e:
    print(f"❌ pop 失败: {e}")

✅ pop 成功，新长度: 5
新结构: Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=5, bias=True)
  (3): ReLU()
  (4): Linear(in_features=30, out_features=5, bias=True)
)


In [8]:
# ============================================================
# 3.1.3 Sequential 的两种创建方式
# ============================================================

# 方式1：直接传入模块（最简洁、最常用）
# 层按传入顺序自动编号：0, 1, 2, ...
seq1 = nn.Sequential(
    nn.Linear(10, 20),   # 第0层
    nn.ReLU()             # 第1层
)

# 方式2：使用 OrderedDict（带名称，便于调试和按名称访问）
# 可以为每一层指定有意义的名称
seq2 = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(10, 20)),   # 名为 fc1 的全连接层
    ('relu', nn.ReLU()),           # 名为 relu 的激活层
    ('fc2', nn.Linear(20, 5))     # 名为 fc2 的全连接层
])
)

# 打印两种方式创建的模型
print(f"方式1: {seq1}")
print(f"方式2 (OrderedDict): {seq2}")


方式1: Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
)
方式2 (OrderedDict): Sequential(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=20, out_features=5, bias=True)
)


In [9]:
# 方式2 可以通过名称直接访问特定层，方便调试
print(f"方式2 可以通过名称访问: {seq2.fc1}")

方式2 可以通过名称访问: Linear(in_features=10, out_features=20, bias=True)


### 3.2 `nn.ModuleList`——灵活控制 forward 逻辑

`nn.Sequential` 代码最简洁，但数据只能固定顺序单向流动，无法实现条件分支、跳跃连接等复杂逻辑。

`nn.ModuleList` 解决了这个问题：它像列表一样存储模块，但 **`forward` 逻辑完全由你控制**。

**优点**：
- ✅ 像列表一样管理模块（`append`、`insert`、`pop`、`extend`）
- ✅ 支持索引访问、切片、迭代
- ✅ forward 逻辑完全由你控制（条件分支、跳跃连接、循环）

`nn.ModuleList` 灵活控制 forward，但没有 `forward` 方法，必须自己写循环或分支逻辑。

**对比：用 Sequential 无法实现的功能，用 ModuleList 可以实现**

In [10]:
# ============================================================
# 3.2.1 对比：Sequential 无法实现条件分支
# ============================================================

print("【Sequential 的限制】")

# Sequential：数据只能按固定顺序单向流动
seq_fixed = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5)
)

x = torch.randn(4, 10)
out = seq_fixed(x)  # 只能按顺序执行，无法跳过 ReLU
print(f"  Sequential 输出: {out.shape}")
print("  无法根据条件跳过某些层\n")


【Sequential 的限制】
  Sequential 输出: torch.Size([4, 5])
  无法根据条件跳过某些层



In [11]:
# ModuleList：可以根据条件选择不同路径
class FlexibleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(10, 5),   # 索引0：路径A
            nn.Linear(10, 3)    # 索引1：路径B
        ])
        self.use_path_a = True  # True走路径A，False走路径B
    
    def forward(self, x):
        if self.use_path_a:
            x = self.layers[0](x)  # 走路径A：10 -> 5
        else:
            x = self.layers[1](x)  # 走路径B：10 -> 3
        return x

model_flex = FlexibleModel()

# 情况1：走路径A
model_flex.use_path_a = True
out1 = model_flex(x)
print(f"  走路径A（10->5）时输出: {out1.shape}")

# 情况2：走路径B
model_flex.use_path_a = False
out2 = model_flex(x)
print(f"  走路径B（10->3）时输出: {out2.shape}")

print("\n✅ 同一个模型，通过条件控制走不同路径，输出维度不同")

  走路径A（10->5）时输出: torch.Size([4, 5])
  走路径B（10->3）时输出: torch.Size([4, 3])

✅ 同一个模型，通过条件控制走不同路径，输出维度不同


In [12]:
# ============================================================
# 3.2.2 nn.ModuleList - 基础使用
# ============================================================

# 定义动态模型，层数由构造函数参数控制
class DynamicModel(nn.Module):
    def __init__(self, num_layers=3):
        super().__init__()
        # 使用 ModuleList 存储多个全连接层
        # 这些层会自动注册到模型，能被 parameters() 识别
        self.layers = nn.ModuleList([
            nn.Linear(10, 20),   # 第1层：10 -> 20
            nn.Linear(20, 30),   # 第2层：20 -> 30
            nn.Linear(30, 5)     # 第3层：30 -> 5
        ])
    
    def forward(self, x):
        # ModuleList 本身不实现 forward 方法
        # 必须自己在 forward 中编写循环逻辑
        # 列表风格：直接迭代子模块
        for layer in self.layers:
            x = layer(x)
        return x

# 实例化模型
model_list = DynamicModel()

# 创建输入数据
x = torch.randn(4, 10)

# 前向传播
print(f"ModuleList: {x.shape} -> {model_list(x).shape}")

# 获取层数
print(f"层数: {len(model_list.layers)}")

# 列表风格：直接迭代子模块（按顺序）
print("\n迭代子模块:")
for i, layer in enumerate(model_list.layers):
    print(f"  层 {i}: {layer.__class__.__name__}")

ModuleList: torch.Size([4, 10]) -> torch.Size([4, 5])
层数: 3

迭代子模块:
  层 0: Linear
  层 1: Linear
  层 2: Linear


In [13]:
# ============================================================
# 3.2.3 ModuleList - 动态操作
# ============================================================

# 创建一个包含1个全连接层的 ModuleList
layers = nn.ModuleList([nn.Linear(10, 20)])
print(f"初始层数: {len(layers)}")


初始层数: 1


In [14]:

# append：在末尾添加层
layers.append(nn.Linear(20, 30))
print(f"append 后: {len(layers)}")


append 后: 2


In [15]:
# insert：在指定位置插入层
# 在索引1的位置插入 Dropout 层，原索引1及之后的层后移
layers.insert(1, nn.Dropout(0.5))
print(f"insert 后: {len(layers)}")
# 打印插入后的结构
print("插入后的结构:")
for i, layer in enumerate(layers):
    print(f"  索引 {i}: {layer.__class__.__name__}")

insert 后: 3
插入后的结构:
  索引 0: Linear
  索引 1: Dropout
  索引 2: Linear


In [16]:
# extend：批量添加多个层
layers.extend([nn.ReLU(), nn.Linear(30, 5)])
print(f"extend 后: {len(layers)}")

extend 后: 5


In [17]:
# pop：移除指定位置的层（需要传入索引）
# 移除索引3的层
removed = layers.pop(3)
print(f"pop 后: {len(layers)}，移除索引3: {removed.__class__.__name__}")

pop 后: 4，移除索引3: ReLU


### 3.3 `nn.ModuleDict`——按名称管理分支

`nn.ModuleList` 灵活控制 forward，但用数字索引访问模块，在多分支/多任务网络中容易混淆。

`nn.ModuleDict` 解决了这个问题：用有意义的名称代替数字索引。

**优点**：
- ✅ 按名称访问模块，代码可读性极高
- ✅ 支持 `.keys()`、`.values()`、`.items()` 等字典操作
- ✅ 支持动态修改（`update`、`pop`）

`nn.ModuleDict` 按名称管理分支，可读性高，但没有 `forward` 方法，必须自己实现分支选择逻辑。

**对比：用 ModuleList 容易混淆，用 ModuleDict 清晰明了**

In [18]:
# ============================================================
# 3.3.1 对比：ModuleList vs ModuleDict 可读性
# ============================================================

print("【ModuleList：数字索引容易混淆】")

# 用 ModuleList 管理多任务头
heads_list = nn.ModuleList([
    nn.Linear(128, 10),   # 索引0：分类头
    nn.Linear(128, 1)     # 索引1：回归头
])

x = torch.randn(4, 128)

# 使用数字索引 —— 容易忘记 0 和 1 分别代表什么
out_class = heads_list[0](x)
out_reg = heads_list[1](x)
print(f"  heads_list[0] 输出: {out_class.shape}   ← 这是分类还是回归？")
print(f"  heads_list[1] 输出: {out_reg.shape}     ← 需要记住索引含义\n")

print("【ModuleDict：名称即文档】")

# 用 ModuleDict 管理多任务头
heads_dict = nn.ModuleDict({
    'classification': nn.Linear(128, 10),
    'regression': nn.Linear(128, 1)
})

# 使用名称访问 —— 一目了然
out_class = heads_dict['classification'](x)
out_reg = heads_dict['regression'](x)
print(f"  heads_dict['classification'] 输出: {out_class.shape}")
print(f"  heads_dict['regression'] 输出: {out_reg.shape}")
print("  ✅ 名称自解释，代码可读性更高")

【ModuleList：数字索引容易混淆】
  heads_list[0] 输出: torch.Size([4, 10])   ← 这是分类还是回归？
  heads_list[1] 输出: torch.Size([4, 1])     ← 需要记住索引含义

【ModuleDict：名称即文档】
  heads_dict['classification'] 输出: torch.Size([4, 10])
  heads_dict['regression'] 输出: torch.Size([4, 1])
  ✅ 名称自解释，代码可读性更高


In [19]:
# ============================================================
# 3.3.2 nn.ModuleDict - 基础使用（不同分支不同维度）
# ============================================================

# 定义多分支网络，每个分支有不同的输出维度
class MultiBranchNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用 ModuleDict 管理多个分支
        # 每个分支有不同的输出维度，模拟不同任务
        self.branches = nn.ModuleDict({
            'classifier': nn.Linear(128, 10),     # 分类分支：输出10类
            'regressor': nn.Linear(128, 1),       # 回归分支：输出1个值
            'embedding': nn.Linear(128, 64),      # 嵌入分支：输出64维特征
            'multi_label': nn.Linear(128, 20),    # 多标签分支：输出20个标签
        })
    
    def forward(self, x, branch_name):
        # 根据分支名称选择对应的处理路径
        # 使用字典的键访问对应的模块 —— 比索引 [0] 可读性高得多
        return self.branches[branch_name](x)

# 实例化多分支网络
model_dict = MultiBranchNet()

# 创建输入数据：4个样本，每个128维
x = torch.randn(4, 128)

# 字典风格：默认迭代键名
print("可用分支（迭代键名）:")
for name in model_dict.branches:
    out_dim = model_dict.branches[name].out_features
    print(f"  {name}: 输出维度 = {out_dim}")


可用分支（迭代键名）:
  classifier: 输出维度 = 10
  regressor: 输出维度 = 1
  embedding: 输出维度 = 64
  multi_label: 输出维度 = 20


In [20]:
print("\n" + "="*60)
print("测试不同分支:")
print("="*60)

# 分别测试每个分支
for branch_name in model_dict.branches.keys():
    output = model_dict(x, branch_name)
    print(f"  {branch_name:15s}: {x.shape} -> {output.shape}")


测试不同分支:
  classifier     : torch.Size([4, 128]) -> torch.Size([4, 10])
  regressor      : torch.Size([4, 128]) -> torch.Size([4, 1])
  embedding      : torch.Size([4, 128]) -> torch.Size([4, 64])
  multi_label    : torch.Size([4, 128]) -> torch.Size([4, 20])


In [21]:
# ============================================================
# 3.3.3 ModuleDict - 综合测试：走不同分支
# ============================================================

# 创建更复杂的多分支网络，每个分支包含多个层
class ComplexMultiBranchNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 共享的特征提取层
        self.shared = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        # 不同的任务分支，每个分支结构不同
        self.branches = nn.ModuleDict({
            'classification': nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 10),
                nn.Softmax(dim=1)
            ),
            'regression': nn.Sequential(
                nn.Linear(128, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            ),
            'feature_extract': nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 32)
            )
        })
    
    def forward(self, x, branch_name):
        # 先通过共享层
        x = self.shared(x)
        # 再通过对应的分支
        return self.branches[branch_name](x)

# 实例化复杂多分支网络
complex_model = ComplexMultiBranchNet()

# 创建输入数据
x = torch.randn(4, 256)

print("复杂多分支网络测试:")
print("="*60)

# 字典风格：迭代键名，展示每个分支的结构和输出
for branch_name in complex_model.branches.keys():
    output = complex_model(x, branch_name)
    print(f"\n分支: {branch_name}")
    print(f"  输入形状: {x.shape}")
    print(f"  输出形状: {output.shape}")
    print(f"  分支结构:")
    for i, layer in enumerate(complex_model.branches[branch_name]):
        print(f"    层 {i}: {layer.__class__.__name__}")

# 统计总参数量
total_params = sum(p.numel() for p in complex_model.parameters())
print(f"\n总参数量: {total_params:,}")

复杂多分支网络测试:

分支: classification
  输入形状: torch.Size([4, 256])
  输出形状: torch.Size([4, 10])
  分支结构:
    层 0: Linear
    层 1: ReLU
    层 2: Linear
    层 3: Softmax

分支: regression
  输入形状: torch.Size([4, 256])
  输出形状: torch.Size([4, 1])
  分支结构:
    层 0: Linear
    层 1: ReLU
    层 2: Linear

分支: feature_extract
  输入形状: torch.Size([4, 256])
  输出形状: torch.Size([4, 32])
  分支结构:
    层 0: Linear
    层 1: ReLU
    层 2: Linear

总参数量: 72,811


In [22]:
# ============================================================
# 3.3.4 ModuleDict - 常用操作
# ============================================================

# 创建 ModuleDict，包含三个任务头
branches = nn.ModuleDict({
    'classifier': nn.Linear(128, 10),     # 分类头：输出10类
    'regressor': nn.Linear(128, 1),       # 回归头：输出1个值
    'feature_extractor': nn.Linear(128, 64)  # 特征提取器
})

# keys()：获取所有键名
print(f"keys(): {list(branches.keys())}")

# values()：获取所有模块
print(f"values(): {list(branches.values())}")

# items()：同时获取键名和模块
print("items():")
for name, module in branches.items():
    print(f"  {name}: {module}")

# 注意：ModuleDict 没有 .get() 方法，使用 in 操作符检查键是否存在
print(f"'classifier' in branches: {'classifier' in branches}")
print(f"'nonexistent' in branches: {'nonexistent' in branches}")

# 安全获取：先检查键是否存在，再取值
classifier_module = branches['classifier'] if 'classifier' in branches else None
print(f"安全获取 'classifier': {classifier_module is not None}")

# pop：移除指定键的模块
removed = branches.pop('regressor')
print(f"pop 后: {list(branches.keys())}")

# update：更新已有的键或添加新的键值对
branches.update({'new_branch': nn.Linear(128, 32)})
print(f"update 后: {list(branches.keys())}")

keys(): ['classifier', 'regressor', 'feature_extractor']
values(): [Linear(in_features=128, out_features=10, bias=True), Linear(in_features=128, out_features=1, bias=True), Linear(in_features=128, out_features=64, bias=True)]
items():
  classifier: Linear(in_features=128, out_features=10, bias=True)
  regressor: Linear(in_features=128, out_features=1, bias=True)
  feature_extractor: Linear(in_features=128, out_features=64, bias=True)
'classifier' in branches: True
'nonexistent' in branches: False
安全获取 'classifier': True
pop 后: ['classifier', 'feature_extractor']
update 后: ['classifier', 'feature_extractor', 'new_branch']


### 3.4 `nn.ParameterList`——管理可训练参数

`nn.ModuleDict` 按名称管理分支，可读性高，但存储的是 `nn.Module` 子类（有 `forward` 方法）。

但如果你想**自定义运算**（非标准层），需要自己管理**原始参数**（权重矩阵、偏置向量）。

`nn.ParameterList` 解决了这个问题：让普通张量变成可训练参数，并像列表一样管理。

**优点**：
- ✅ 把普通张量变成可训练参数（自动注册到 `_parameters`）
- ✅ 支持 `append`、`insert`、`pop` 等列表操作
- ✅ 自动参与梯度计算、设备迁移、状态管理

`nn.ParameterList` 让普通张量变成可训练参数，但没有 `forward` 方法，必须自己实现运算逻辑；用数字索引访问，参数多时容易混淆。

**对比：用 ModuleList 存储网络层 vs 用 ParameterList 存储原始参数**

In [23]:
# ============================================================
# 3.4.1 对比：普通张量 vs ParameterList
# ============================================================

print("【普通张量：不会被训练】")

class NormalTensorModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ❌ 普通张量：不会注册为可训练参数
        self.weight = torch.randn(10, 5)
    
    def forward(self, x):
        return x @ self.weight

model_normal = NormalTensorModel()
print(f"  model.parameters() 识别的参数数量: {len(list(model_normal.parameters()))}")
print("  ❌ 普通张量不会被注册，优化器无法更新\n")

print("【ParameterList：可训练参数】")

class ParameterModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ✅ ParameterList：自动注册为可训练参数
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(10, 5))
        ])
    
    def forward(self, x):
        return x @ self.weights[0]

model_param = ParameterModel()
print(f"  model.parameters() 识别的参数数量: {len(list(model_param.parameters()))}")
print("  ✅ ParameterList 中的参数被自动注册，优化器可以更新")
print("  ✅ 自动参与梯度计算、设备迁移、模型保存")

【普通张量：不会被训练】
  model.parameters() 识别的参数数量: 0
  ❌ 普通张量不会被注册，优化器无法更新

【ParameterList：可训练参数】
  model.parameters() 识别的参数数量: 1
  ✅ ParameterList 中的参数被自动注册，优化器可以更新
  ✅ 自动参与梯度计算、设备迁移、模型保存


In [24]:
# ============================================================
# 3.4.2 nn.ParameterList - 基础使用
# ============================================================

# 定义使用 ParameterList 管理参数的模型
class ParameterModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用 ParameterList 存储多个权重矩阵
        # 注意：必须用 nn.Parameter 包装张量，否则不会被注册
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(10, 20)),   # 权重1：10x20
            nn.Parameter(torch.randn(20, 5))     # 权重2：20x5
        ])
        # 使用 ParameterList 存储多个偏置向量
        self.biases = nn.ParameterList([
            nn.Parameter(torch.zeros(20)),       # 偏置1：20维
            nn.Parameter(torch.zeros(5))         # 偏置2：5维
        ])
    
    def forward(self, x):
        # 手动实现两层全连接网络
        # 列表风格：直接迭代参数
        for i, w in enumerate(self.weights):
            # 矩阵乘法：x @ w，然后加上偏置
            x = x @ w + self.biases[i]
            # 使用 ReLU 激活
            x = F.relu(x)
        return x

# 实例化模型
model_params = ParameterModel()

# 创建输入数据
x = torch.randn(4, 10)

# 前向传播
print(f"ParameterList: {x.shape} -> {model_params(x).shape}")

# 统计总参数量
total_params = sum(p.numel() for p in model_params.parameters())
print(f"总参数量: {total_params:,}")

# 列表风格：直接迭代参数
print("\n迭代参数:")
for i, w in enumerate(model_params.weights):
    print(f"  权重{i+1}: {w.shape}")
for i, b in enumerate(model_params.biases):
    print(f"  偏置{i+1}: {b.shape}")

ParameterList: torch.Size([4, 10]) -> torch.Size([4, 5])
总参数量: 325

迭代参数:
  权重1: torch.Size([10, 20])
  权重2: torch.Size([20, 5])
  偏置1: torch.Size([20])
  偏置2: torch.Size([5])


In [25]:
# ============================================================
# 3.4.3 ParameterList - 参数注册验证
# ============================================================

# 重新实例化模型
model_params = ParameterModel()

# 验证参数是否被正确注册：model.parameters() 应该能识别所有参数
params_list = list(model_params.parameters())
print(f"model.parameters() 识别的参数数量: {len(params_list)}")
print(f"是否被识别: {len(params_list) > 0}")

# 打印每个参数的名称和形状
print("\n参数名称:")
for name, param in model_params.named_parameters():
    print(f"  {name}: {param.shape}")

# 验证梯度是否正常计算
x = torch.randn(4, 10)
output = model_params(x)
loss = output.sum()  # 计算损失（所有元素求和）
loss.backward()      # 反向传播，计算梯度

# 检查每个参数是否都有梯度
print("\n梯度验证:")
for name, param in model_params.named_parameters():
    if param.grad is not None:
        print(f"  {name}: 梯度存在，形状 {param.grad.shape}")
    else:
        print(f"  {name}: 梯度不存在")

model.parameters() 识别的参数数量: 4
是否被识别: True

参数名称:
  weights.0: torch.Size([10, 20])
  weights.1: torch.Size([20, 5])
  biases.0: torch.Size([20])
  biases.1: torch.Size([5])

梯度验证:
  weights.0: 梯度存在，形状 torch.Size([10, 20])
  weights.1: 梯度存在，形状 torch.Size([20, 5])
  biases.0: 梯度存在，形状 torch.Size([20])
  biases.1: 梯度存在，形状 torch.Size([5])


### 3.5 `nn.ParameterDict`——按名称管理可训练参数

`nn.ParameterList` 让普通张量变成可训练参数，但用数字索引访问，参数多时容易混淆。

`nn.ParameterDict` 解决了这个问题：按名称管理可训练参数。

**优点**：
- ✅ 按名称访问参数，代码可读性极高
- ✅ 按名称保存/加载预训练权重，避免顺序错乱
- ✅ 支持 `.keys()`、`.values()`、`.items()`、`update`、`pop` 等操作

**对比：用 ParameterList 容易混淆，用 ParameterDict 清晰明了**

In [26]:
# ============================================================
# 3.5.1 对比：ParameterList vs ParameterDict 可读性
# ============================================================

print("【ParameterList：数字索引容易混淆】")

# 用 ParameterList 管理参数
params_list = nn.ParameterList([
    nn.Parameter(torch.randn(10, 20)),   # 索引0：是什么？
    nn.Parameter(torch.randn(20, 5)),    # 索引1：是什么？
    nn.Parameter(torch.zeros(20)),       # 索引2：是什么？
    nn.Parameter(torch.zeros(5))         # 索引3：是什么？
])

x = torch.randn(4, 10)

# 使用数字索引 —— 很难记住每个索引的含义
x = x @ params_list[0] + params_list[2]  # 0 和 2 是什么？
x = F.relu(x)
x = x @ params_list[1] + params_list[3]  # 1 和 3 是什么？
print(f"  params_list[0], [1], [2], [3] 分别代表什么？容易混淆\n")

print("【ParameterDict：名称即文档】")

# 用 ParameterDict 管理参数
params_dict = nn.ParameterDict({
    'weight1': nn.Parameter(torch.randn(10, 20)),
    'weight2': nn.Parameter(torch.randn(20, 5)),
    'bias1': nn.Parameter(torch.zeros(20)),
    'bias2': nn.Parameter(torch.zeros(5))
})

# 使用名称访问 —— 一目了然
x = torch.randn(4, 10)
x = x @ params_dict['weight1'] + params_dict['bias1']
x = F.relu(x)
x = x @ params_dict['weight2'] + params_dict['bias2']
print(f"  params_dict['weight1'], ['weight2'], ['bias1'], ['bias2']")
print("  ✅ 名称自解释，代码可读性更高")

【ParameterList：数字索引容易混淆】
  params_list[0], [1], [2], [3] 分别代表什么？容易混淆

【ParameterDict：名称即文档】
  params_dict['weight1'], ['weight2'], ['bias1'], ['bias2']
  ✅ 名称自解释，代码可读性更高


In [27]:
# ============================================================
# 3.5.2 nn.ParameterDict - 基础使用
# ============================================================

# 定义使用 ParameterDict 管理参数的模型
class NamedParameterModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用 ParameterDict 存储带名称的参数
        # 每个参数都有一个有意义的键名
        self.params = nn.ParameterDict({
            'weight1': nn.Parameter(torch.randn(10, 20)),   # 权重1
            'weight2': nn.Parameter(torch.randn(20, 5)),    # 权重2
            'bias1': nn.Parameter(torch.zeros(20)),         # 偏置1
            'bias2': nn.Parameter(torch.zeros(5))           # 偏置2
        })
    
    def forward(self, x):
        # 通过名称访问参数，代码更清晰易读
        x = x @ self.params['weight1'] + self.params['bias1']
        x = F.relu(x)
        x = x @ self.params['weight2'] + self.params['bias2']
        return x

# 实例化模型
model_named = NamedParameterModel()

# 创建输入数据
x = torch.randn(4, 10)

# 前向传播
print(f"ParameterDict: {x.shape} -> {model_named(x).shape}")

# 字典风格：默认迭代键名
print("\n可用参数（迭代键名）:")
for name in model_named.params:
    print(f"  {name}: {model_named.params[name].shape}")

# 使用 .items() 同时获取键名和参数
print("\n使用 .items():")
for name, param in model_named.params.items():
    print(f"  {name}: {param.shape}")

ParameterDict: torch.Size([4, 10]) -> torch.Size([4, 5])

可用参数（迭代键名）:
  bias1: torch.Size([20])
  bias2: torch.Size([5])
  weight1: torch.Size([10, 20])
  weight2: torch.Size([20, 5])

使用 .items():
  bias1: torch.Size([20])
  bias2: torch.Size([5])
  weight1: torch.Size([10, 20])
  weight2: torch.Size([20, 5])


In [28]:
# ============================================================
# 3.5.3 ParameterDict - 常用操作
# ============================================================

# 创建 ParameterDict，包含多个参数
params = nn.ParameterDict({
    'weight1': nn.Parameter(torch.randn(10, 20)),
    'bias1': nn.Parameter(torch.zeros(20)),
    'weight2': nn.Parameter(torch.randn(20, 5))
})

# keys()：获取所有键名
print(f"keys(): {list(params.keys())}")

# values()：获取所有参数
print(f"values(): {list(params.values())}")

# items()：同时获取键名和参数
print("items():")
for name, param in params.items():
    print(f"  {name}: {param.shape}")

# 注意：ParameterDict 没有 .get() 方法，使用 in 操作符检查键是否存在
print(f"'weight1' in params: {'weight1' in params}")
print(f"'nonexistent' in params: {'nonexistent' in params}")

# 安全获取：先检查键是否存在，再取值
weight1_param = params['weight1'] if 'weight1' in params else None
print(f"安全获取 'weight1': {weight1_param is not None}")

# 通过键名直接访问参数
print(f"params['weight2'].shape = {params['weight2'].shape}")

# 动态添加新参数：直接赋值即可
params['weight3'] = nn.Parameter(torch.randn(5, 2))
print(f"动态添加后: {list(params.keys())}")

# pop：移除指定键的参数
removed = params.pop('bias1')
print(f"pop 后: {list(params.keys())}")

keys(): ['bias1', 'weight1', 'weight2']
values(): [Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       requires_grad=True), Parameter containing:
tensor([[ 0.6378,  0.0305,  0.0606, -0.2950,  1.3040, -0.5657, -0.6046, -0.0110,
         -0.4160, -0.0678, -0.6666,  0.5570,  2.1366,  0.3719, -0.2841, -0.3426,
          0.3331, -1.2824, -0.4285, -0.0300],
        [-1.3903,  1.0536, -1.9024,  0.4993,  0.1036, -0.8672,  1.5637,  0.6119,
          0.1663, -0.9654,  1.0548, -0.1041,  1.2315,  0.2568, -0.3465,  1.5005,
         -0.8432,  0.8908,  0.4746,  0.0650],
        [ 0.2944, -0.4859,  2.1960,  0.2727, -0.4857,  0.8139,  0.1135,  0.0361,
          0.6490,  0.9595, -0.0096,  0.5427,  1.0712,  0.7262, -0.2662, -0.7580,
         -0.4718,  1.1438,  0.8832, -0.4421],
        [-1.0898, -1.5747,  0.5548, -0.9405,  0.5679, -1.6300, -0.6226,  0.7478,
         -1.7604, -1.3114,  1.0155,  0.1135, -0.6746,  0.2084, -1.5064,  0.1176,
  

### 3.6 递进总结：五种容器的演进路径

```
Sequential (最基础)
    ↓ 需要灵活控制 forward 逻辑
ModuleList
    ↓ 需要按名称管理分支
ModuleDict
    ↓ 需要管理可训练参数（非标准层）
ParameterList
    ↓ 需要按名称管理可训练参数
ParameterDict (最灵活)
```

| 容器 | 解决什么问题 | 何时使用 |
|------|------------|---------|
| `Sequential` | 省代码 | 简单顺序网络 |
| `ModuleList` | 灵活控制 forward | 动态结构、条件分支 |
| `ModuleDict` | 按名称管理分支 | 多分支、多任务网络 |
| `ParameterList` | 把张量变成可训练参数 | 自定义运算层 |
| `ParameterDict` | 按名称管理可训练参数 | 需要按名称保存/加载参数 |

In [29]:
# ============================================================
# 3.6.1 验证 ParameterList 的自动求导
# ============================================================

class SimpleParamModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 用 ParameterList 存储可训练参数
        # nn.Parameter 默认 requires_grad=True
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(10, 20)),
            nn.Parameter(torch.randn(20, 5))
        ])
        self.bias = nn.Parameter(torch.zeros(5))
    
    def forward(self, x):
        x = x @ self.weights[0]
        x = F.relu(x)
        x = x @ self.weights[1] + self.bias
        return x

print("="*60)
print("【验证 ParameterList 自动求导】")
print("="*60)

# 创建模型和数据
model = SimpleParamModel()
x = torch.randn(4, 10, requires_grad=True)  # 输入也开启梯度

# 前向传播
output = model(x)
loss = output.sum()

# 反向传播 —— 自动求导！
loss.backward()

# 验证：所有 Parameter 都有梯度
print("\n各参数梯度状态:")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name}: ✅ 梯度存在，形状 {param.grad.shape}")
    else:
        print(f"  {name}: ❌ 梯度不存在")

# 输入 x 也有梯度（因为 requires_grad=True）
print(f"\n  x.grad: {'✅ 存在' if x.grad is not None else '❌ 不存在'}")
print(f"  x.grad 形状: {x.grad.shape if x.grad is not None else 'N/A'}")

print("\n【结论】ParameterList 构建的网络完全支持自动求导！")

【验证 ParameterList 自动求导】

各参数梯度状态:
  bias: ✅ 梯度存在，形状 torch.Size([5])
  weights.0: ✅ 梯度存在，形状 torch.Size([10, 20])
  weights.1: ✅ 梯度存在，形状 torch.Size([20, 5])

  x.grad: ✅ 存在
  x.grad 形状: torch.Size([4, 10])

【结论】ParameterList 构建的网络完全支持自动求导！


In [30]:
# ============================================================
# 3.6.2 容器对比示例：Module容器 vs Parameter容器
# ============================================================

print("="*60)
print("【Module 容器 vs Parameter 容器】")
print("="*60)

# 1. Module容器：存储网络层（有forward）
print("\n1. Module容器（存储 nn.Module 子类）:")
print("-"*40)

# 使用 ModuleList 存储全连接层
fc_layers = nn.ModuleList([
    nn.Linear(10, 20),
    nn.Linear(20, 5)
])

# 这些层有 forward 方法，可以直接调用
x = torch.randn(4, 10)
for layer in fc_layers:
    x = layer(x)  # 调用 forward
print(f"  ModuleList 输出形状: {x.shape}")

# 2. Parameter容器：存储参数（无forward）
print("\n2. Parameter容器（存储 nn.Parameter 张量）:")
print("-"*40)

# 使用 ParameterList 存储权重和偏置
weights = nn.ParameterList([
    nn.Parameter(torch.randn(10, 20)),
    nn.Parameter(torch.randn(20, 5))
])
biases = nn.ParameterList([
    nn.Parameter(torch.zeros(20)),
    nn.Parameter(torch.zeros(5))
])

# 这些 Parameter 没有 forward 方法
# 需要手动进行矩阵运算
x = torch.randn(4, 10)
for i in range(len(weights)):
    x = x @ weights[i] + biases[i]  # 手动运算
    x = F.relu(x)
print(f"  ParameterList 输出形状: {x.shape}")

print("\n【关键区别】")
print("  Module容器: 存储有 forward 方法的网络层，可以直接调用")
print("  Parameter容器: 存储无 forward 方法的参数张量，需要手动运算")

【Module 容器 vs Parameter 容器】

1. Module容器（存储 nn.Module 子类）:
----------------------------------------
  ModuleList 输出形状: torch.Size([4, 5])

2. Parameter容器（存储 nn.Parameter 张量）:
----------------------------------------
  ParameterList 输出形状: torch.Size([4, 5])

【关键区别】
  Module容器: 存储有 forward 方法的网络层，可以直接调用
  Parameter容器: 存储无 forward 方法的参数张量，需要手动运算


In [31]:
# ============================================================
# 3.6.3 ParameterList 典型应用：可学习位置编码
# ============================================================

class LearnablePositionalEncoding(nn.Module):
    """可学习的位置编码——Transformer 中常用"""
    def __init__(self, max_len, d_model):
        super().__init__()
        # 为什么用 ParameterList？
        # 因为位置编码是一组可学习的向量，数量由 max_len 决定
        # 如果不用 ParameterList，需要写 max_len 个 nn.Parameter 属性，不现实
        self.pos_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(d_model)) for _ in range(max_len)
        ])
    
    def forward(self, x):
        # 每个位置加上对应的可学习编码
        for i in range(x.size(1)):
            x[:, i, :] = x[:, i, :] + self.pos_embeddings[i]
        return x

# 实例化并测试
pos_encoding = LearnablePositionalEncoding(max_len=10, d_model=64)
x = torch.randn(4, 10, 64)  # batch=4, seq_len=10, d_model=64
output = pos_encoding(x)

print("="*60)
print("【ParameterList 典型应用：可学习位置编码】")
print("="*60)
print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"位置编码数量: {len(pos_encoding.pos_embeddings)}")
print(f"每个编码维度: {pos_encoding.pos_embeddings[0].shape}")

# 验证参数是否被注册
params_count = sum(p.numel() for p in pos_encoding.parameters())
print(f"总参数量: {params_count:,}")
print("\n参数名称:")
for name, param in pos_encoding.named_parameters():
    print(f"  {name}: {param.shape}")

【ParameterList 典型应用：可学习位置编码】
输入形状: torch.Size([4, 10, 64])
输出形状: torch.Size([4, 10, 64])
位置编码数量: 10
每个编码维度: torch.Size([64])
总参数量: 640

参数名称:
  pos_embeddings.0: torch.Size([64])
  pos_embeddings.1: torch.Size([64])
  pos_embeddings.2: torch.Size([64])
  pos_embeddings.3: torch.Size([64])
  pos_embeddings.4: torch.Size([64])
  pos_embeddings.5: torch.Size([64])
  pos_embeddings.6: torch.Size([64])
  pos_embeddings.7: torch.Size([64])
  pos_embeddings.8: torch.Size([64])
  pos_embeddings.9: torch.Size([64])


In [32]:
# ============================================================
# 3.6.4 实际应用场景对比
# ============================================================

print("="*60)
print("【实际应用场景对比】")
print("="*60)

# 场景1：ModuleList 适用于构建可变层数的网络
print("\n场景1：使用 ModuleList 构建可变深度网络")
print("-"*40)

class VariableDepthNet(nn.Module):
    def __init__(self, num_layers):
        super().__init__()
        # ModuleList 存储的是完整的网络层
        self.layers = nn.ModuleList([
            nn.Linear(128, 128) for _ in range(num_layers)
        ])
        self.output = nn.Linear(128, 10)
    
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.output(x)

net = VariableDepthNet(num_layers=5)
x = torch.randn(4, 128)
print(f"  VariableDepthNet: {x.shape} -> {net(x).shape}")
print(f"  层数: {len(net.layers)}")

# 场景2：ParameterList 适用于管理自定义的权重矩阵
print("\n场景2：使用 ParameterList 管理自定义权重")
print("-"*40)

class CustomWeightNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ParameterList 存储的是纯参数，不是网络层
        # 适合实现自定义的运算逻辑
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(128, 256)),
            nn.Parameter(torch.randn(256, 128)),
        ])
        # 注意：这里没有 forward 方法，需要手动实现运算
    
    def forward(self, x):
        # 手动实现矩阵乘法
        x = x @ self.weights[0]
        x = F.relu(x)
        x = x @ self.weights[1]
        return x

custom_net = CustomWeightNet()
x = torch.randn(4, 128)
print(f"  CustomWeightNet: {x.shape} -> {custom_net(x).shape}")
print(f"  参数量: {sum(p.numel() for p in custom_net.parameters()):,}")

print("\n【选择建议】")
print("  需要标准网络层（Linear, Conv2d 等） -> 用 ModuleList/ModuleDict/Sequential")
print("  需要自定义参数运算（非标准层） -> 用 ParameterList/ParameterDict")
print("  需要参数带有名称便于加载保存 -> 用 ParameterDict")

【实际应用场景对比】

场景1：使用 ModuleList 构建可变深度网络
----------------------------------------
  VariableDepthNet: torch.Size([4, 128]) -> torch.Size([4, 10])
  层数: 5

场景2：使用 ParameterList 管理自定义权重
----------------------------------------
  CustomWeightNet: torch.Size([4, 128]) -> torch.Size([4, 128])
  参数量: 65,536

【选择建议】
  需要标准网络层（Linear, Conv2d 等） -> 用 ModuleList/ModuleDict/Sequential
  需要自定义参数运算（非标准层） -> 用 ParameterList/ParameterDict
  需要参数带有名称便于加载保存 -> 用 ParameterDict


## 4. 容器使用场景与选择指南

### 4.1 5种容器各自存在的必要性（一句话总结）

| 容器 | 存在的必要性（一句话） | 核心价值 |
|------|----------------------|---------|
| **`nn.Sequential`** | 让你**不用写 forward 循环**，数据自动按顺序流过各层 | **省代码** |
| **`nn.ModuleList`** | 让你能**像列表一样管理模块**，但 forward 逻辑自己控制 | **省管理** |
| **`nn.ModuleDict`** | 让你能**按名称管理分支**，在多任务/多分支网络中可读性最高 | **省脑子** |
| **`nn.ParameterList`** | 让你能**把多个张量变成可训练参数并管理**，用于自定义运算 | **让普通张量变成可训练参数** |
| **`nn.ParameterDict`** | 让你能**按名称管理可训练参数**，方便加载预训练权重 | **让可训练参数有名字** |

### 4.2 容器底层实现汇总

| 容器类 | 继承自 | 内部存储 | 存储内容 | 注册到 |
|-------|--------|---------|---------|--------|
| `nn.Sequential` | `nn.Module` | `ModuleList` 或 `OrderedDict` | `nn.Module` 子类 | `_modules` |
| `nn.ModuleList` | `nn.Module` | Python `list` | `nn.Module` 子类 | `_modules` |
| `nn.ModuleDict` | `nn.Module` | Python `dict` | `nn.Module` 子类 | `_modules` |
| `nn.ParameterList` | `nn.Module` | Python `list` | `nn.Parameter` | `_parameters` |
| `nn.ParameterDict` | `nn.Module` | Python `dict` | `nn.Parameter` | `_parameters` |

### 4.3 所有容器的核心特性对比

| 容器 | 是否有 `forward` | 是否可迭代 | 迭代返回 | 是否可动态修改 | 访问方式 |
|------|-----------------|-----------|---------|---------------|---------|
| `nn.Sequential` | ✅ | ✅ | 子模块（按顺序） | ✅ | 索引 `[i]` |
| `nn.ModuleList` | ❌ | ✅ | 子模块（按顺序） | ✅ | 索引 `[i]` |
| `nn.ModuleDict` | ❌ | ✅（迭代键） | 键名 | ✅ | 键 `['key']` |
| `nn.ParameterList` | ❌ | ✅ | 参数（按顺序） | ✅ | 索引 `[i]` |
| `nn.ParameterDict` | ❌ | ✅（迭代键） | 键名 | ✅ | 键 `['key']` |

### 4.4 快速选择指南

**第一维度：存储内容不同**

| 容器类型 | 存储内容 | 继承自 | 注册到 | 典型用途 |
|---------|---------|--------|--------|---------|
| `nn.Sequential` | `nn.Module` 子类 | `nn.Module` | `_modules` | 组织网络层 |
| `nn.ModuleList` | `nn.Module` 子类 | `nn.Module` | `_modules` | 组织网络层 |
| `nn.ModuleDict` | `nn.Module` 子类 | `nn.Module` | `_modules` | 组织网络层 |
| `nn.ParameterList` | `nn.Parameter` | `nn.Module` | `_parameters` | 管理可训练参数 |
| `nn.ParameterDict` | `nn.Parameter` | `nn.Module` | `_parameters` | 管理可训练参数 |

**核心区别**：
- **前三个（Module 容器）**：存储的是**网络层/模块**（如 Linear、Conv2d），有 forward 方法
- **后两个（Parameter 容器）**：存储的是**可训练参数**（张量），没有 forward 方法

**第二维度：按场景选择**

| 场景 | 推荐容器 | 原因 |
|------|---------|------|
| 简单前馈网络 | `Sequential` | 代码最简洁，顺序执行 |
| 动态循环网络 | `ModuleList` | 可动态增减层，支持循环 |
| 多分支/多任务网络 | `ModuleDict` | 通过名称管理不同分支 |
| 自定义参数管理 | `ParameterList` | 管理多个可学习参数 |
| 带名称的参数集 | `ParameterDict` | 方便加载和保存 |

### 4.5 一句话道出 Module 容器与 Parameter 容器的区别

> **`ModuleList`/`ModuleDict`/`Sequential` 是用来装"网络层"的（有 `forward` 方法，能直接 `layer(x)` 调用），而 `ParameterList`/`ParameterDict` 是用来装"参数张量"的（没有 `forward` 方法，需要手动运算 `x @ w + b`）。**

**更通俗地说：**

| | Module 容器 | Parameter 容器 |
|---|---|---|
| **装的是什么** | 完整的"零件"（如 Linear、Conv2d） | 原材料"螺丝钉"（如权重矩阵、偏置向量） |
| **怎么用** | 直接调用：`layer(x)` | 手动组装：`x @ weight + bias` |
| **有无 forward** | 有 | 无 |
| **类比** | 整机厂买来的"电机" | 自己采购的"铜线圈" |

**一个直观的例子：**

```python
# ModuleList：装的是"完整的层"，拿来就能用
self.layers = nn.ModuleList([nn.Linear(10, 20)])  # Linear 自带 forward
x = self.layers[0](x)  # ✅ 直接调用

# ParameterList：装的是"原始参数"，需要自己写运算逻辑
self.weights = nn.ParameterList([nn.Parameter(torch.randn(10, 20))])  # 只是个张量
x = x @ self.weights[0]  # ✅ 手动运算
```

所以：
- **想搭积木（标准层）→ 用 Module 容器**
- **想造积木（自定义参数）→ 用 Parameter 容器**

### 4.6 Sequential vs ModuleList 的区别

两者都存储模块、都可索引、都可迭代、都可动态修改，但核心区别：

| 特性 | `nn.Sequential` | `nn.ModuleList` |
|------|----------------|-----------------|
| 有 `forward` | ✅ 自动按顺序执行 | ❌ 需自己实现 |
| 可动态修改 | ✅ | ✅ |
| 代码简洁度 | ✅ 最简洁 | ❌ 需写循环 |
| 适用场景 | 固定/动态结构网络（有顺序） | 动态结构网络（需控制 forward 逻辑） |

```python
# Sequential：自动 forward
model = nn.Sequential(nn.Linear(10, 20), nn.ReLU())
out = model(x)  # ✅ 一行 forward

# ModuleList：手动 forward
self.layers = nn.ModuleList([nn.Linear(10, 20), nn.ReLU()])
for layer in self.layers:
    x = layer(x)  # 必须自己写循环
```

### 4.7 ParameterList/ParameterDict 存在的价值

一句话总结：**当你想把"原始张量"变成"可训练参数"并让 PyTorch 自动管理（梯度、设备、保存加载）时，就用它们。**

**核心价值：让普通张量"升级"为可训练参数**

```python
# ❌ 普通张量：不会被训练
w = torch.randn(10, 20)  # 只是普通数据，没有梯度

# ✅ Parameter：可训练参数
w = nn.Parameter(torch.randn(10, 20))  # 有梯度，会被优化器更新
```

**但问题是**：如果你有**多个**这样的自定义参数怎么办？

```python
# ❌ 用 Python 列表存多个 Parameter——不会注册！
self.weights = [nn.Parameter(torch.randn(10, 20)), 
                nn.Parameter(torch.randn(20, 5))]  # 模型找不到这些参数！

# ✅ 用 ParameterList——自动注册！
self.weights = nn.ParameterList([
    nn.Parameter(torch.randn(10, 20)),
    nn.Parameter(torch.randn(20, 5))
])  # model.parameters() 能识别，优化器能更新
```

**典型使用场景**：

| 场景 | 用 ParameterList/Dict 的原因 | 能否用 Module 替代？ |
|------|------------------------------|---------------------|
| 自定义运算层 | 实现非标准运算（如自定义注意力、特殊矩阵运算） | ❌ 没有现成的 Module |
| 动态参数数量 | 参数个数在运行时才能确定 | ❌ 普通属性无法动态增减 |
| 需要命名参数 | 方便按名称加载预训练权重 | ❌ 普通属性没有命名机制 |
| 参数共享/复用 | 多个地方使用同一组参数 | ⚠️ 可以但不够灵活 |

### 4.8 ParameterList/ParameterDict 支持自动求导吗？

**是的！完全支持自动求导。**

> **`ParameterList` 和 `ParameterDict` 里装的 `nn.Parameter` 是自带 `requires_grad=True` 的张量，PyTorch 的自动求导机制对它们一视同仁。**

**为什么能自动求导？**

`nn.Parameter` 本质就是 `torch.Tensor` 的子类，默认开启了 `requires_grad=True`：

```python
w = nn.Parameter(torch.randn(10, 20))
print(f"requires_grad: {w.requires_grad}")  # True
print(f"是否是 Tensor: {isinstance(w, torch.Tensor)}")  # True
```

**所以**：
- `ParameterList` 装的是 `Parameter` → 有梯度
- `Parameter` 就是 `Tensor` → 有自动求导
- 因此 → **完全支持自动求导**

**与 Module 容器的对比**：

| | Module 容器 | Parameter 容器 |
|---|---|---|
| **存储内容** | `nn.Module` 子类（如 Linear） | `nn.Parameter`（张量） |
| **内部参数** | Linear 内部也有 `weight` 和 `bias`（都是 Parameter） | 直接就是 Parameter |
| **是否自动求导** | ✅ 是（因为内部是 Parameter） | ✅ 是（因为本身就是 Parameter） |
| **本质** | 包装好的零件 | 裸露的原材料 |

**结论**：殊途同归，最终都是 `Parameter` 在参与梯度计算。

### 4.9 容器核心要点总结

| 要点 | 说明 |
|------|------|
| 自动注册 | 容器内模块/参数自动注册到父模块的 `_modules` 或 `_parameters` |
| 设备迁移 | `.to(device)` 自动迁移所有内容 |
| 状态管理 | `state_dict()` 自动包含所有内容 |
| 梯度管理 | 所有参数自动参与梯度计算 |
| 可迭代性 | 所有容器都实现了迭代协议，但返回内容不同 |
| 动态修改 | **所有容器都支持**动态增删改操作 |
| 不可用原生列表 | Python 列表/字典不会触发注册机制 |

### 4.10 常见错误与陷阱

| 错误 | 后果 | 正确做法 |
|------|------|---------|
| 使用 Python 列表存 Module | 参数不注册，优化器不更新 | 用 `ModuleList` |
| 使用 Python 字典存 Module | 参数不注册，优化器不更新 | 用 `ModuleDict` |
| 在 forward 中动态创建层 | 每次重建，效率低且不注册 | 在 `__init__` 中创建并存入容器 |
| 嵌套容器过深 | 难以调试和维护 | 保持扁平或合理分层 |
| 用 Python 列表存 Parameter | 参数不注册，无法训练 | 用 `ParameterList` |
| 普通张量直接放入容器 | 不会被识别为可训练参数 | 用 `nn.Parameter()` 包装 |
| 混淆容器的迭代方式 | 字典风格容器默认迭代键，不是值 | 用 `.items()` 或 `.values()` |

### 4.11 综合示例：容器综合应用

以下示例在一个模型中同时使用多种容器，展示它们如何协同工作。

In [33]:
# ============================================================
# 综合示例：在一个模型中同时使用多种容器
# ============================================================

class ComprehensiveModel(nn.Module):
    def __init__(self, num_blocks=3):
        super().__init__()
        
        # 1. Sequential：基础特征提取模块
        # 三层全连接网络，按顺序执行
        self.feature_extractor = nn.Sequential(
            nn.Linear(128, 256),   # 输入128维，输出256维
            nn.ReLU(),              # 激活函数
            nn.Linear(256, 256)    # 保持256维
        )
        
        # 2. ModuleList：多个残差块（动态数量）
        # 使用列表推导式创建 num_blocks 个全连接层
        self.res_blocks = nn.ModuleList([
            nn.Linear(256, 256) for _ in range(num_blocks)
        ])
        
        # 3. ModuleDict：多任务头
        # 不同的任务使用不同的输出层
        self.task_heads = nn.ModuleDict({
            'classification': nn.Linear(256, 10),   # 分类任务：输出10类
            'regression': nn.Linear(256, 1)         # 回归任务：输出1个值
        })
        
        # 4. ParameterDict：自定义可学习参数
        # 用于对最终输出进行缩放和偏移
        self.custom_params = nn.ParameterDict({
            'scale': nn.Parameter(torch.ones(1)),   # 缩放因子，初始为1
            'bias': nn.Parameter(torch.zeros(1))    # 偏置项，初始为0
        })
    
    def forward(self, x, task='classification'):
        # 步骤1：通过特征提取器
        x = self.feature_extractor(x)
        
        # 步骤2：通过残差块（带跳跃连接）
        for block in self.res_blocks:
            residual = x          # 保存输入作为残差
            x = block(x)          # 通过当前块
            x = x + residual      # 残差连接：输出 = 变换 + 原始输入
        
        # 步骤3：根据任务选择对应的任务头
        x = self.task_heads[task](x)
        
        # 步骤4：应用自定义参数进行缩放和偏移
        x = x * self.custom_params['scale'] + self.custom_params['bias']
        return x

# 实例化综合模型，包含3个残差块
model = ComprehensiveModel(num_blocks=3)

# 创建输入数据
x = torch.randn(4, 128)

# 打印模型结构，展示使用的容器类型
print("模型结构:")
print(f"  feature_extractor: {type(model.feature_extractor).__name__}")
print(f"  res_blocks: {type(model.res_blocks).__name__} (长度={len(model.res_blocks)})")
print(f"  task_heads: {type(model.task_heads).__name__} (键={list(model.task_heads.keys())})")
print(f"  custom_params: {type(model.custom_params).__name__} (键={list(model.custom_params.keys())})")

# 测试不同任务
print("\n测试不同任务:")
for task in ['classification', 'regression']:
    output = model(x, task)
    print(f"  {task}: {x.shape} -> {output.shape}")

# 统计总参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params:,}")

模型结构:
  feature_extractor: Sequential
  res_blocks: ModuleList (长度=3)
  task_heads: ModuleDict (键=['classification', 'regression'])
  custom_params: ParameterDict (键=['bias', 'scale'])

测试不同任务:
  classification: torch.Size([4, 128]) -> torch.Size([4, 10])
  regression: torch.Size([4, 128]) -> torch.Size([4, 1])

总参数量: 299,021
